In [18]:
import numpy as np
import pandas as pd

from footix.data_io.footballdata import ScrapFootballData
from footix.models.bayesian import BayesianModel
from footix.strategy.over_under_bets import OverUnderOddsInput
from footix.strategy.over_under_selection import (
    select_bets_by_edge_only,
)
from footix.strategy.over_under_staking import apply_stakes_kelly

In [19]:
df = ScrapFootballData(
    competition="FRA Ligue 2", season="2024-2025", path="./data", force_reload=True
).get_fixtures()

In [20]:
start_pct = 0.6
step = 9
start_point = int(len(df) * start_pct)
bankroll = 100.0
history_bankroll = []

for start_idx in range(start_point, len(df) - step + 1, step):
    train_dataset = df.iloc[:start_idx]
    test_dataset = df.iloc[start_idx : start_idx + step]

    model = BayesianModel(n_goals=20, n_teams=18, calibrate=True)
    model.fit(train_dataset)

    list_over_under_odds = []
    dico_score_matrix = []
    results_map = {}

    for _, match in test_dataset.iterrows():
        goal_matrix = model.predict(home_team=match["home_team"], away_team=match["away_team"])
        odds_u_25 = match["b365<2.5"]
        odds_o_25 = match["b365>2.5"]
        match_id = f"{match['home_team']} {match['away_team']}"

        fthg = match.get("fthg", np.nan)
        ftag = match.get("ftag", np.nan)
        total_goals = None
        if pd.notna(fthg) and pd.notna(ftag):
            total_goals = int(fthg) + int(ftag)

        list_over_under_odds.append(
            OverUnderOddsInput(
                match_id=match_id,
                home_team=match["home_team"],
                away_team=match["away_team"],
                over_odds=odds_o_25,
                under_odds=odds_u_25,
                actual_goals=total_goals,
            )
        )
        dico_score_matrix.append(goal_matrix)
        results_map[match_id] = total_goals

    bets = select_bets_by_edge_only(
        odds_inputs=list_over_under_odds, goal_matrices=dico_score_matrix
    )
    bets = apply_stakes_kelly(bets=bets, bankroll=bankroll, max_stake_pct=0.3)

    day_pnl = 0.0
    for bet in bets:
        actual_goals = results_map.get(bet.match_id)
        if actual_goals is None or pd.isna(actual_goals):
            continue
        bet.actual_goals = int(actual_goals)
        profit = bet.profit()
        if profit is None:
            continue
        day_pnl += profit

    bankroll += day_pnl
    history_bankroll.append(
        {
            "start_idx": start_idx,
            "pnl": day_pnl,
            "bankroll": bankroll,
            "n_bets": len(bets),
        }
    )

history_df = pd.DataFrame(history_bankroll)
history_df

Running window adaptation


<div><progress max="1000" value="1000"></progress> 100.00% [1000/1000 00:00&lt;?]</div>

<div><progress max="1000" value="1000"></progress> 100.00% [1000/1000 00:00&lt;?]</div>

<div><progress max="1000" value="1000"></progress> 100.00% [1000/1000 00:00&lt;?]</div>

<div><progress max="1000" value="1000"></progress> 100.00% [1000/1000 00:00&lt;?]</div>

...

<div><progress max="2000" value="2000"></progress> 100.00% [2000/2000 00:00&lt;?]</div>

<div><progress max="2000" value="2000"></progress> 100.00% [2000/2000 00:00&lt;?]</div>

<div><progress max="2000" value="2000"></progress> 100.00% [2000/2000 00:00&lt;?]</div>

<div><progress max="2000" value="2000"></progress> 100.00% [2000/2000 00:00&lt;?]</div>

,start_idx,pnl,bankroll,n_bets
0,183,-28.567753,71.432247,9
1,192,8.796990,80.229236,5
2,201,0.811968,81.041204,2
3,210,-7.958327,73.082878,5
4,219,10.329414,83.412291,5
5,228,8.383309,91.795600,5
6,237,3.381369,95.176969,6
7,246,1.416012,96.592981,6
8,255,-16.811116,79.781866,6
9,264,-4.866221,74.915645,3


In [21]:
bankroll

np.float64(68.65143954292776)